### **CIFAR-10 dataset**

In [ ]:
import tensorflow as tf
import numpy as np

# Load the CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train, y_train = x_train[:5000].reshape(5000,-1)/255., y_train[:5000]
x_test, y_test = x_test[:1000].reshape(1000,-1)/255., y_test[:1000]

print(x_train.shape, x_train.dtype)
print(y_train.shape, y_train.dtype)
print(x_test.shape, x_test.dtype)
print(y_test.shape, y_test.dtype)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
(5000, 3072) float64
(5000, 1) uint8
(1000, 3072) float64
(1000, 1) uint8


In [ ]:
# softmax cross-entropy loss
def loss(x, y, W, b):
  y_ = np.exp( (W @ x.T + b).T )
  s_ = np.sum(y_, axis=1)
  y_ = np.take_along_axis(y_, y, axis=1)
  y_ = y_ / s_[:,np.newaxis]
  return np.mean( - np.log(y_) )

# compute accuracy
def acc(x, y, W, b):
  y_ = (W @ x.T + b).T
  y_ = np.argmax(y_, axis=1)
  return np.mean(y_ == y[:,0])

best_loss = np.inf
best_W, best_b = None, None
for i in range(1000):
  W = np.random.normal(loc=0.0, scale=0.001, size=(10,32*32*3))
  b = np.random.normal(loc=0.0, scale=0.001, size=(10,1))

  l = loss(x_train, y_train, W, b)
  if l < best_loss:
    best_loss = l;
    best_W = W
    best_b = b
    print(i, loss(x_train, y_train, W, b), acc(x_train, y_train, W, b))

print('---')
print(loss(x_train, y_train, best_W, best_b), acc(x_train, y_train, best_W, best_b))
print(loss(x_test, y_test, best_W, best_b), acc(x_test, y_test, best_W, best_b))

0 2.3009198865852567 0.102
24 2.3008927083205495 0.1208
92 2.299102355541851 0.1072
371 2.298893030618162 0.131
---
2.298893030618162 0.131
2.297306343720123 0.131


In [ ]:
def gradient_descent(x, y, W, b, lr):
  y_one_hot = np.eye(10)[y[:,0]]

  y_ = np.exp( (W @ x.T + b).T )
  s_ = np.sum(y_, axis=1)[:,np.newaxis]
  p = y_/s_

  grad_W = ( (p - y_one_hot).T @ x ) / len(x)
  grad_b = np.mean(p - y_one_hot, axis=0)[:,np.newaxis]

  W -= lr*grad_W
  b -= lr*grad_b

  return W, b

W = np.random.normal(loc=0.0, scale=0.001, size=(10,32*32*3))
b = np.random.normal(loc=0.0, scale=0.001, size=(10,1))

best_loss = np.inf
best_W, best_b = None, None
for i in range(100):
  W, b = gradient_descent(x_train, y_train, W, b, 0.01)

  l = loss(x_train, y_train, W, b)
  if l < best_loss:
    best_loss = l;
    best_W = W
    best_b = b

  print(i, loss(x_train, y_train, W, b), acc(x_train, y_train, W, b))

print('---')
print(loss(x_train, y_train, best_W, best_b), acc(x_train, y_train, best_W, best_b))
print(loss(x_test, y_test, best_W, best_b), acc(x_test, y_test, best_W, best_b))

0 2.2893091186719787 0.1134
1 2.280344763105737 0.1234
2 2.271999584271176 0.1346
3 2.2639668913647584 0.1456
4 2.2562179580107786 0.1632
5 2.2487385303363854 0.1798
6 2.2415158402250666 0.198
7 2.2345377852435027 0.2086
8 2.227792850712677 0.221
9 2.2212700876812645 0.2292
10 2.2149590944265545 0.235
11 2.2088499970249704 0.2426
12 2.202933428883908 0.2478
13 2.1972005095590146 0.2518
14 2.1916428231896194 0.259
15 2.1862523968397256 0.2622
16 2.181021678981997 0.2656
17 2.175943518316162 0.2684
18 2.1710111430727013 0.2716
19 2.1662181409177124 0.273
20 2.1615584395451295 0.2748
21 2.1570262880173843 0.2794
22 2.152616238894831 0.2798
23 2.14832313117707 0.282
24 2.1441420740654045 0.284
25 2.1400684315444503 0.285
26 2.136097807772108 0.2862
27 2.132226033260226 0.2872
28 2.128449151823119 0.2882
29 2.1247634082672526 0.2914
30 2.1211652367927707 0.2936
31 2.117651250075772 0.2946
32 2.1142182289992526 0.2962
33 2.110863113000262 0.2964
34 2.1075829910008776 0.2974
35 2.104375092891

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# compute accuracy
def acc(x, y, model):
  with torch.no_grad():
    y_ = model(x)
  y_ = torch.argmax(y_, dim=1)
  return torch.mean((y_ == y).float())

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3072, 100)
        self.fc2 = nn.Linear(100, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

net = Net().cuda()

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=10)

inputs = torch.from_numpy(x_train).float().cuda()
labels = torch.from_numpy(y_train[:,0]).to(torch.long).cuda()

for i in range(200):
  # zero the parameter gradients
  optimizer.zero_grad()

  # forward + backward + optimize
  outputs = net(inputs)
  loss = criterion(outputs, labels)
  loss.backward()
  optimizer.step()

  print(i, loss.item(), acc(inputs, labels, net), optimizer.param_groups[0]['lr'])

  if i%10 == 9:
    scheduler.step()

print('Finished Training')

0 2.308073043823242 tensor(0.1218, device='cuda:0') 0.01
1 2.3034071922302246 tensor(0.1238, device='cuda:0') 0.01
2 2.2962708473205566 tensor(0.1354, device='cuda:0') 0.01
3 2.2890405654907227 tensor(0.1350, device='cuda:0') 0.01
4 2.2827656269073486 tensor(0.1248, device='cuda:0') 0.01
5 2.2767410278320312 tensor(0.1202, device='cuda:0') 0.01
6 2.270082473754883 tensor(0.1236, device='cuda:0') 0.01
7 2.262416124343872 tensor(0.1338, device='cuda:0') 0.01
8 2.2540295124053955 tensor(0.1552, device='cuda:0') 0.01
9 2.245246171951294 tensor(0.1810, device='cuda:0') 0.01
10 2.2362585067749023 tensor(0.1988, device='cuda:0') 0.0091
11 2.2279911041259766 tensor(0.2088, device='cuda:0') 0.0091
12 2.2196578979492188 tensor(0.2118, device='cuda:0') 0.0091
13 2.2109339237213135 tensor(0.2212, device='cuda:0') 0.0091
14 2.201678991317749 tensor(0.2298, device='cuda:0') 0.0091
15 2.192049503326416 tensor(0.2366, device='cuda:0') 0.0091
16 2.1823954582214355 tensor(0.2402, device='cuda:0') 0.0091